## Step 1 — Load features + labels

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
import json, os, pickle

data_dir    = Path().resolve().parent / 'data' / 'features'
outputs_dir = Path().resolve().parent / 'outputs'
os.makedirs(outputs_dir, exist_ok=True)

features = pd.read_parquet(data_dir / 'features.parquet')
labels   = pd.read_parquet(data_dir / 'labels.parquet')

# Cast condition_cluster — parquet preserves category dtype from Stage 2
features['condition_cluster'] = features['condition_cluster'].astype(str)

df = features.merge(labels, on='member_id')
print(f"Combined dataset: {df.shape}")
print(f"Positive class:   {df['high_acute_risk'].mean():.1%}")

## Step 2 — Prepare feature matrix

In [ ]:
FEATURE_COLS = [
    'age', 'age_band', 'gender', 'plan_type', 'employer_group_size', 'tenure_months',
    'has_msk_flag', 'has_metabolic_flag', 'has_mh_flag',
    'comorbidity_count', 'condition_cluster',
    'total_claims_6m', 'total_spend_6m',
    'gp_visits_6m', 'specialist_visits_6m', 'allied_health_claims_6m',
    'days_since_last_allied', 'allied_health_utilisation_rate',
    'specialist_to_gp_ratio', 'zero_allied_health_flag', 'high_gp_low_allied',
    'sessions_remaining_physio', 'sessions_remaining_chiro',
    'sessions_remaining_dietetics', 'sessions_remaining_psychology',
    'any_benefits_remaining', 'benefit_utilisation_rate',
    # Interaction features (Lever 2)
    'msk_zero_allied', 'metabolic_zero_allied', 'mh_zero_allied',
    'comorbid_zero_allied',
    'age_zero_allied', 'bronze_high_comorbid',
]
CAT_COLS = ['age_band', 'gender', 'plan_type', 'employer_group_size', 'condition_cluster']

# Leakage guard — label must not be in FEATURE_COLS
assert 'high_acute_risk' not in FEATURE_COLS, "LEAKAGE: label found in FEATURE_COLS"

X = df[FEATURE_COLS].copy()
y = df['high_acute_risk'].copy()

for col in CAT_COLS:
    X[col] = X[col].astype('category')

print(f"X shape: {X.shape}")
print(f"y positive rate: {y.mean():.1%}")

## Step 3 — Train / validation / test split (60 / 20 / 20)

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

print(f"Train:      {len(X_train):,}  (positive: {y_train.mean():.1%})")
print(f"Validation: {len(X_val):,}  (positive: {y_val.mean():.1%})")
print(f"Test:       {len(X_test):,}  (positive: {y_test.mean():.1%})")

## Step 4 — Calculate scale_pos_weight

In [ ]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos
print(f"scale_pos_weight (from train split): {spw:.2f}  ({neg:,} neg / {pos:,} pos)")

# Cross-check SPW against the labels file rather than a hardcoded constant.
# The positive class rate changed after the DGP redesign in Stage 3.
_lb = pd.read_parquet(data_dir / 'labels.parquet')
_expected_spw = (_lb['high_acute_risk'] == 0).sum() / _lb['high_acute_risk'].sum()
assert abs(spw - _expected_spw) < 0.10, (
    f"scale_pos_weight from train split ({spw:.2f}) diverges from "
    f"labels file ({_expected_spw:.2f}) — check stratified split"
)
print(f"✓ scale_pos_weight {spw:.2f} consistent with labels file ({_expected_spw:.2f})")

## Step 5 — Create LightGBM datasets

In [ ]:
train_data = lgb.Dataset(
    X_train, label=y_train,
    categorical_feature=CAT_COLS,
    free_raw_data=False)
val_data = lgb.Dataset(
    X_val, label=y_val,
    categorical_feature=CAT_COLS,
    reference=train_data,
    free_raw_data=False)

print("LightGBM datasets created")

## Step 6 — Train

In [ ]:
params = {
    'objective':         'binary',
    'metric':            ['auc', 'binary_logloss'],  # AUC first → early stopping tracks AUC
    'learning_rate':     0.01,                        # finer steps → more iterations before plateau
    'num_leaves':        31,                          # was 63 — reduce overfitting capacity
    'min_child_samples': 50,                          # was 10 — force conservative splits
    'scale_pos_weight':  spw,
    'feature_fraction':  0.7,                         # was 0.8 — more subsampling randomness
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'reg_alpha':         0.1,                         # was 0.01 — stronger L1
    'reg_lambda':        0.1,                         # was 0.01 — stronger L2
    'verbose':           -1,
    'n_jobs':            -1,
    'seed':              42,
}

model = lgb.train(
    params,
    train_set=train_data,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'val'],
    num_boost_round=1000,
    callbacks=[
        lgb.early_stopping(100, first_metric_only=True,verbose=True),
        lgb.log_evaluation(50),
    ],
)

print(f"\nBest iteration: {model.best_iteration}")

# ── Training summary ──────────────────────────────────────────
_sep = '─' * 52
train_auc_best = model.best_score['train']['auc']
val_auc_best   = model.best_score['val']['auc']
gap            = train_auc_best - val_auc_best
gap_label      = 'healthy' if gap < 0.05 else 'mild overfit'
print('\n' + _sep)
print('  TRAINING SUMMARY')
print(_sep)
print('  Best iteration     {:>6,}'.format(model.best_iteration))
print('  Train AUC (best)   {:>6.4f}'.format(train_auc_best))
print('  Val AUC   (best)   {:>6.4f}'.format(val_auc_best))
print('  Train / Val gap    {:>6.4f}   {}'.format(gap, gap_label))
print(_sep)

## Step 7 — Evaluate on test set

In [ ]:
y_pred_proba = model.predict(X_test, num_iteration=model.best_iteration)

roc_auc = roc_auc_score(y_test, y_pred_proba)
pr_auc  = average_precision_score(y_test, y_pred_proba)

# Business metric: Recall @ top 20%
test_df = pd.DataFrame({'y_true': y_test.values, 'y_pred': y_pred_proba})
test_df = test_df.sort_values('y_pred', ascending=False).reset_index(drop=True)
top20   = test_df.head(int(len(test_df) * 0.20))

recall_top20    = top20['y_true'].sum() / y_test.sum()
precision_top20 = top20['y_true'].mean()

# ── Results table ─────────────────────────────────────────────
def _met(val, target):
    return 'met' if val >= target else 'BELOW TARGET'

_sep = '─' * 58
print('\n' + _sep)
print('  {:<26}  {:>8}  {:>8}  {}'.format('METRIC', 'VALUE', 'TARGET', 'STATUS'))
print(_sep)
print('  {:<26}  {:>8.4f}  {:>8}  {}'.format('ROC-AUC', roc_auc, '> 0.750', _met(roc_auc, 0.750)))
print('  {:<26}  {:>8.4f}  {:>8}  {}'.format('PR-AUC', pr_auc, '> 0.550', _met(pr_auc, 0.550)))
print('  {:<26}  {:>8.4f}  {:>8}  {}'.format('Recall @ top 20%', recall_top20, '> 0.650', _met(recall_top20, 0.650)))
print('  {:<26}  {:>8.4f}  {:>8}  {}'.format('Precision @ top 20%', precision_top20, '> 0.350', _met(precision_top20, 0.350)))
print(_sep)

metrics = {
    'roc_auc':              round(roc_auc, 4),
    'pr_auc':               round(pr_auc, 4),
    'recall_top20pct':      round(recall_top20, 4),
    'precision_top20pct':   round(precision_top20, 4),
    'best_iteration':       model.best_iteration,
    'positive_class_rate':  round(float(y.mean()), 4),
    'scale_pos_weight':     round(spw, 2),
    'train_size':           len(X_train),
    'val_size':             len(X_val),
    'test_size':            len(X_test),
}

## Step 7b — Isotonic calibration

Maps raw LightGBM scores to calibrated probabilities using isotonic regression
fitted on the validation set. Ranking metrics (ROC-AUC, Recall@top20%) are
unaffected — calibration is monotone. PR-AUC and Precision@top20% may shift
as score magnitudes become more meaningful for threshold-based business rules.

In [ ]:
from sklearn.isotonic import IsotonicRegression

# Fit on validation set — must not touch test set to avoid leakage
y_val_pred = model.predict(X_val, num_iteration=model.best_iteration)
calibrator = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(y_val_pred, y_val)

# Apply to test predictions
y_pred_calibrated = calibrator.predict(y_pred_proba)

# Calibrated metrics
roc_auc_cal = roc_auc_score(y_test, y_pred_calibrated)
pr_auc_cal  = average_precision_score(y_test, y_pred_calibrated)

test_cal_df = pd.DataFrame({'y_true': y_test.values, 'y_pred': y_pred_calibrated})
test_cal_df = test_cal_df.sort_values('y_pred', ascending=False).reset_index(drop=True)
top20_cal   = test_cal_df.head(int(len(test_cal_df) * 0.20))
recall_cal    = top20_cal['y_true'].sum() / y_test.sum()
precision_cal = top20_cal['y_true'].mean()

# Comparison table
_sep = '─' * 62
print('\n' + _sep)
print('  CALIBRATION COMPARISON')
print(_sep)
print('  {:<26}  {:>10}  {:>10}'.format('Metric', 'Raw', 'Calibrated'))
print(_sep)
print('  {:<26}  {:>10.4f}  {:>10.4f}'.format('ROC-AUC', roc_auc, roc_auc_cal))
print('  {:<26}  {:>10.4f}  {:>10.4f}'.format('PR-AUC', pr_auc, pr_auc_cal))
print('  {:<26}  {:>10.4f}  {:>10.4f}'.format('Recall @ top 20%', recall_top20, recall_cal))
print('  {:<26}  {:>10.4f}  {:>10.4f}'.format('Precision @ top 20%', precision_top20, precision_cal))
print(_sep)
print('  Score range raw:         {:.4f} — {:.4f}'.format(y_pred_proba.min(), y_pred_proba.max()))
print('  Score range calibrated:  {:.4f} — {:.4f}'.format(y_pred_calibrated.min(), y_pred_calibrated.max()))
print(_sep)

# Append calibrated metrics to the metrics dict for saving in Step 9
metrics.update({
    'roc_auc_calibrated':            round(roc_auc_cal, 4),
    'pr_auc_calibrated':             round(pr_auc_cal, 4),
    'recall_top20pct_calibrated':    round(recall_cal, 4),
    'precision_top20pct_calibrated': round(precision_cal, 4),
})

## Step 8 — SHAP feature importance

In [ ]:
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print("Calculating SHAP values...")
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Newer SHAP returns a single 2D array for binary classifiers; older returns a list
if isinstance(shap_values, list):
    shap_pos = shap_values[1]
else:
    shap_pos = shap_values

shap.summary_plot(shap_pos, X_test, show=False, max_display=15)
plt.tight_layout()
plt.savefig(outputs_dir / 'shap_summary.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"Saved: {outputs_dir / 'shap_summary.png'}")

feat_importance = pd.DataFrame({
    'feature':        FEATURE_COLS,
    'mean_abs_shap':  np.abs(shap_pos).mean(axis=0),
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

feat_importance.to_csv(outputs_dir / 'feature_importance.csv', index=False)

total_shap = feat_importance['mean_abs_shap'].sum()
feat_importance['signal_pct'] = feat_importance['mean_abs_shap'] / total_shap * 100

_sep = '─' * 60
print('\n' + _sep)
print('  FEATURE IMPORTANCE  (mean |SHAP|, top 15 of {})'.format(len(feat_importance)))
print(_sep)
print('  {:<4}  {:<34}  {:>8}  {:>9}'.format('#', 'Feature', 'SHAP', 'Signal %'))
print(_sep)
for rank, (_, row) in enumerate(feat_importance.head(15).iterrows(), 1):
    print('  {:<4}  {:<34}  {:>8.4f}  {:>8.1f}%'.format(rank, row['feature'], row['mean_abs_shap'], row['signal_pct']))
print(_sep)
print('  Top 15 capture  {:.1f}% of total SHAP signal'.format(feat_importance.head(15)['signal_pct'].sum()))
print(_sep)

## Step 9 — Save model and metrics

In [ ]:
with open(outputs_dir / 'model.pkl', 'wb') as f:
    pickle.dump(model, f)
print(f"Saved: {outputs_dir / 'model.pkl'}")

with open(outputs_dir / 'calibrator.pkl', 'wb') as f:
    pickle.dump(calibrator, f)
print(f"Saved: {outputs_dir / 'calibrator.pkl'}")

with open(outputs_dir / 'eval_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"Saved: {outputs_dir / 'eval_metrics.json'}")

# Quick load test
with open(outputs_dir / 'model.pkl', 'rb') as f:
    m_test = pickle.load(f)
test_preds = m_test.predict(X_test.head(3)).round(3)
print(f"Load test predictions: {test_preds}")
print("Model load test passed ✓")

## Validation Checks

In [ ]:
from pathlib import Path
import pickle, json

outputs_dir = Path().resolve().parent / 'outputs'

with open(outputs_dir / 'model.pkl', 'rb') as f:
    m = pickle.load(f)
assert m.best_iteration > 0, "Model has no iterations — training did not complete"

with open(outputs_dir / 'eval_metrics.json') as f:
    saved = json.load(f)

assert saved['roc_auc']         >= 0.70, f"ROC-AUC too low: {saved['roc_auc']}"
assert saved['pr_auc']          >= 0.35, f"PR-AUC too low: {saved['pr_auc']}"
assert saved['recall_top20pct'] >= 0.35, f"Recall@top20% too low: {saved['recall_top20pct']}"
assert 'high_acute_risk' not in FEATURE_COLS, "LEAKAGE: label in feature list"
assert (outputs_dir / 'shap_summary.png').exists(), "SHAP plot missing"
assert (outputs_dir / 'feature_importance.csv').exists(), "Feature importance CSV missing"

print("✅ All Stage 4 validation checks passed")
print(f"ROC-AUC={saved['roc_auc']},  PR-AUC={saved['pr_auc']},  Recall@top20%={saved['recall_top20pct']}")
print(f"Best iteration: {saved['best_iteration']},  scale_pos_weight: {saved['scale_pos_weight']}")

In [ ]:
# ── Bootstrap 95% Confidence Intervals ──
# Runs 1,000 bootstrap resamples of the test set for uncertainty estimates
import numpy as np
from sklearn.metrics import precision_recall_curve, auc as auc_fn
rng = np.random.default_rng(42)
n_boot, n_test = 1000, len(y_test)
boot_aucs, boot_praucs, boot_recalls = [], [], []
for _ in range(n_boot):
    idx = rng.integers(0, n_test, size=n_test)
    yt, yp = y_test.iloc[idx], y_pred_proba[idx]
    boot_aucs.append(roc_auc_score(yt, yp))
    prec, rec, _ = precision_recall_curve(yt, yp)
    boot_praucs.append(auc_fn(rec, prec))
    top_k = int(len(yp) * 0.2)
    top_idx = np.argsort(yp)[-top_k:]
    boot_recalls.append(yt.iloc[top_idx].mean())
def ci95(vals): return np.percentile(vals, [2.5, 97.5])
auc_lo, auc_hi = ci95(boot_aucs)
prauc_lo, prauc_hi = ci95(boot_praucs)
recall_lo, recall_hi = ci95(boot_recalls)
print('Bootstrap 95% CI (n=1,000):')
print(f'  ROC-AUC:       {eval_metrics["roc_auc"]:.4f}  [{auc_lo:.4f}, {auc_hi:.4f}]')
print(f'  PR-AUC:        {eval_metrics["pr_auc"]:.4f}  [{prauc_lo:.4f}, {prauc_hi:.4f}]')
print(f'  Recall@top20%: {eval_metrics["recall_top20pct"]:.4f}  [{recall_lo:.4f}, {recall_hi:.4f}]')
